# Who is my neighbour, when there is no map?

Both halves of this notebook deal with dependence that geography does not
supply. In the first, the connections are a social network and the question is
whether the effect of one's peers can be told apart from the effect of being the
sort of person who has those peers. In the second, there is no network at all —
units move together because something unobserved moves all of them, and the
question is whether that something is spatial dependence or a common factor.

The two failures are mirror images. A peer-effects model on the wrong network
structure is **unidentified** and will still return numbers. A spatial model
fitted to factor-driven data is **identified but wrong**, and will report a
plausible `rho`. Each section ends with the diagnostic that distinguishes the
case, because in both the estimator itself stays silent.

In [ ]:
import Pkg
Pkg.activate(joinpath(@__DIR__, ".."))
Pkg.instantiate()

using SpatialEconometrics
using LinearAlgebra, SparseArrays, Statistics, Printf, Random
import StatsAPI: residuals

## 1. The reflection problem, as a matter of rank

Manski's reflection problem is usually stated as a worry. On a network it is
something sharper: a rank condition, which either holds or does not, and can be
checked before any estimation.

The linear-in-means model is

    y = a + b*G*y + X*gamma + G*X*delta + e

where `G` is the row-standardised network. Identification needs `X`, `G*X` and
`G*G*X` to be linearly independent. In **equal-sized** groups where everyone
interacts with everyone else, `G*G*X` is a linear combination of the first two,
so the rank fails and `b` cannot be separated from `delta`. Make the groups
*unequal* and the rank condition is restored — the same model, the same volume
of data, identified purely because group sizes differ.

Bramoullé, Djebbari and Fortin (2009) is the result; `peer_identification`
reports it.

In [ ]:
println("three rooms of five -- everyone interacts with everyone:")
show(stdout, peer_identification(group_weights([5, 5, 5])))
println("\n")
println("rooms of four, five and six -- same model, unequal sizes:")
show(stdout, peer_identification(group_weights([4, 5, 6])))

## 2. Peer effects on a network that does identify them

A sparse random graph has the irregularity that equal-sized groups lack, so the
rank condition holds. The truth below is `b = 0.4` for the endogenous peer
effect, with contextual effects `delta` alongside, and the estimator should
recover both.

Two things in the output are worth more attention than the point estimates. The
**first-stage F** says whether the instruments — built from `G*G*X` — carry any
information; a rank condition that holds in exact arithmetic can still be nearly
violated in practice. The **Sargan** statistic tests the overidentifying
restrictions, which is the only internal check available that the instrument set
is consistent with the model.

In [ ]:
RNG = MersenneTwister(2026)

function random_network(rng, n, d)
    for _ in 1:500
        e = [(i, j) for i in 1:n for j in (i + 1):n if rand(rng) < d]
        isempty(e) && continue
        G = network_weights(e, n)
        all(!iszero, sum(G, dims = 2)) && return G
    end
    error("no connected draw")
end

# y = a + b*Gy + X*gamma + G*X*delta + e
function simulate_peers(rng, G, b, gamma, delta; a = 1.0)
    n = size(G, 1); Gd = Matrix(G)
    X = randn(rng, n, length(gamma))
    y = (Matrix(1.0I, n, n) - b * Gd) \
        (a .+ X * gamma .+ Gd * X * delta .+ randn(rng, n))
    return y, X
end

B, GAM, DEL = 0.4, [1.0, -0.5], [0.8, 0.3]

G = random_network(RNG, 300, 0.04)
y, X = simulate_peers(RNG, G, B, GAM, DEL)
f = fit_peer_effects(y, X, G)

@printf("endogenous peer effect b : %+.4f   [truth %+.2f]\n", f.beta, B)
@printf("gamma                    : %s   [truth %s]\n",
        join([@sprintf("%+.4f", g) for g in f.gamma], " "),
        join([@sprintf("%+.2f", g) for g in GAM], " "))
@printf("delta                    : %s   [truth %s]\n",
        join([@sprintf("%+.4f", d) for d in f.delta], " "),
        join([@sprintf("%+.2f", d) for d in DEL], " "))
println()
@printf("first-stage F   : %.2f\n", f.first_stage_F)
@printf("Sargan          : %.3f on %d df, p = %.3f\n",
        f.sargan, f.sargan_df, f.sargan_p)

### Is anything left in the residuals?

A peer-effects model accounts for dependence running through `G`. It says
nothing about dependence running through anything else — a shared location, a
shared teacher, a shared supplier. `network_moran` applies Moran's I to the
fitted residuals under a *second* weight matrix, which is how that question gets
asked: if the residuals are still correlated under a structure the model did not
use, the model is incomplete in a specific, nameable way.

Below, the same network is used, which should find nothing because that
dependence has already been modelled. A second matrix representing a different
structure is where this diagnostic earns its keep.

In [ ]:
nm = network_moran(f, Matrix(G))

@printf("Moran I of the residuals under G : %+.4f\n", nm.I_stat)
@printf("expected under the null          : %+.4f\n", nm.E_I)
@printf("z                                : %+.4f\n", nm.z_stat)
@printf("p-value                          : %.4f\n", nm.p_value)
println()
println("Nothing left under G, which is what a correctly specified peer model")
println("should show. The test is only informative against a structure the")
println("estimator did not already absorb.")

## 3. A time effect is a factor with equal loadings

Now a panel with no network. Each period, something unobserved shifts every
unit:

    y_t = (I - rho*W)^-1 * (X_t*beta + lambda .* f_t + e_t)

When the loadings `lambda` are **equal**, that factor is just a time effect, and
a two-way fixed-effects estimator removes it. When the loadings **differ across
units**, no set of additive dummies can absorb it, and the cross-sectional
correlation it induces looks like spatial dependence to an estimator that has
only `rho` with which to explain correlation.

The comparison below holds everything else fixed and changes only the loadings.
`rho` is 0.4 in both.

In [ ]:
function simulate_factor(rng, W, N, T, rho, beta, loadings)
    Im   = Matrix(1.0I, N, N)
    Ainv = inv(Im - rho * Matrix(W))
    X    = randn(rng, N * T, length(beta))
    f    = randn(rng, T)
    lam  = loadings(rng, N)
    y    = Vector{Float64}(undef, N * T)
    for t in 1:T
        r = ((t - 1) * N + 1):(t * N)
        y[r] = Ainv * (X[r, :] * beta .+ lam .* f[t] .+ randn(rng, N))
    end
    return y, X
end

Wf = make_rook_weights(7); Nf = size(Wf, 1); Tf = 30
RHO, BETA = 0.4, [1.0, -0.5]

designs = (("equal loadings (a time effect)",    (rng, n) -> ones(n)),
           ("heterogeneous loadings (a factor)", (rng, n) -> 1.0 .+ 3.0 .* randn(rng, n)))

@printf("%-36s %10s %10s %+9s\n", "loadings", "rho-hat", "truth", "error")
for (lab, ld) in designs
    yf, Xf = simulate_factor(MersenneTwister(99), Wf, Nf, Tf, RHO, BETA, ld)
    m = fit_spatial_panel(SAR(), TwoWayFE(), Wf, Xf, yf, Nf, Tf;
                          approach = DirectQML(), method = ExactLU())
    @printf("%-36s %10.4f %10.4f %+9.4f\n", lab, m.rho, RHO, m.rho - RHO)
end

## 4. Two diagnostics that answer different questions

`pesaran_cd` tests the null of **weak** cross-sectional dependence. It is a
yes-or-no answer.

`alpha_exponent` answers the question that remains: *how strong*. It estimates
the exponent `alpha` in a cross-sectional variance that scales like
`N^(2*alpha - 1)`. `alpha = 1` is a factor touching every unit; `alpha = 0.5` is
the weak-dependence boundary; sparse spatial dependence sits below it.

Below that boundary the exponent is **not identified** — Bailey, Kapetanios and
Pesaran (2016) show the idiosyncratic contribution to the variance of the
cross-sectional mean is at least as large as a weak factor's, so no amount of
data separates them. `alpha_exponent` therefore **throws rather than returning a
number**, and the refusal is itself the answer: the dependence is weak.

That is worth seeing on three inputs — residuals from both designs above, a raw
series driven by a strong factor, and pure noise.

In [ ]:
function pooled_residuals(seed, ld)
    yf, Xf = simulate_factor(MersenneTwister(seed), Wf, Nf, Tf, RHO, BETA, ld)
    m = fit_spatial_panel(SAR(), TwoWayFE(), Wf, Xf, yf, Nf, Tf;
                          approach = DirectQML(), method = ExactLU())
    return reshape(residuals(m), Nf, Tf)
end

function report_alpha(label, v, N, T; kwargs...)
    cd = pesaran_cd(v, N, T)
    @printf("%-42s CD %+8.3f  p %.4f  ", label, cd.stat, cd.p_value)
    try
        ae = alpha_exponent(v, N, T; kwargs...)
        @printf("alpha %.3f  [%.2f, %.2f]\n", ae.alpha, ae.ci[1], ae.ci[2])
    catch err
        println("alpha not identified (refused)")
    end
end

report_alpha("residuals, equal loadings",        vec(pooled_residuals(99, designs[1][2])), Nf, Tf)
report_alpha("residuals, heterogeneous loadings", vec(pooled_residuals(99, designs[2][2])), Nf, Tf)

# A raw series driven by a strong common factor: every unit loads on f.
rngA = MersenneTwister(5)
fA   = randn(rngA, Tf)
lamA = 1.0 .+ 0.3 .* randn(rngA, Nf)
XA   = [lamA[i] * fA[t] + randn(rngA) for i in 1:Nf, t in 1:Tf]
report_alpha("raw series with a strong factor", vec(XA), Nf, Tf; kappa2 = 0.5)

report_alpha("pure noise", vec(randn(MersenneTwister(6), Nf, Tf)), Nf, Tf)

println()
println("Note the two residual rows: CD rejects weak dependence, and the")
println("exponent still cannot be identified. The two diagnostics are not")
println("redundant, and they can disagree in this direction.")

## 5. The estimator that survives a factor

`fit_sp_cce` augments the regression with cross-sectional averages of `y` and
`X`. Those averages proxy for the unobserved factor, so the correlation it
induces is absorbed instead of being attributed to `rho`. Pesaran (2006) is the
common-correlated-effects idea; the spatial version keeps `rho` in the model
while the augmentation handles the factor.

Keep the size of the problem in proportion. On this design the fixed-effects
`rho` was wrong by about 0.03 on a truth of 0.4 — around 8%, not a catastrophe.
The factor bias here is real and modest, and it would grow with the spread of
the loadings and shrink with `T`. The case for CCE is that it costs little and
removes a bias whose size you cannot know in advance, not that fixed effects
collapse.

In [ ]:
yh, Xh = simulate_factor(MersenneTwister(99), Wf, Nf, Tf, RHO, BETA, designs[2][2])

m_fe  = fit_spatial_panel(SAR(), TwoWayFE(), Wf, Xh, yh, Nf, Tf;
                          approach = DirectQML(), method = ExactLU())
m_cce = fit_sp_cce(yh, Xh, Wf, Nf, Tf)

@printf("%-40s %10s %+10s\n", "", "rho-hat", "error")
@printf("%-40s %10.4f %+10.4f\n", "two-way fixed effects", m_fe.rho,  m_fe.rho - RHO)
@printf("%-40s %10.4f %+10.4f\n", "spatial CCE", m_cce.rho, m_cce.rho - RHO)
@printf("%-40s %10.4f\n", "truth", RHO)

## 6. What to carry away

- **The reflection problem is a rank condition, checkable in advance.** Equal
  groups where everyone interacts fail it; unequal groups do not. Same model,
  same data volume, and identification turns on group sizes alone.
  `peer_identification` reports this before anything is estimated, and a
  peer-effects estimator on an unidentified design returns numbers without
  complaint.
- Read the first-stage F and the Sargan p-value next to the peer coefficient.
  A rank condition that holds exactly can still be weak in practice.
- `network_moran` asks whether residual dependence survives under a *different*
  structure from the one modelled. Under the same `G` it should find nothing, as
  it does here; its value is against a second matrix the estimator never saw.
- **A factor and a time effect are the same object with different loadings.**
  Equal loadings are absorbed by fixed effects, so `rho` came back at 0.4022.
  Heterogeneous loadings are not, and `rho` fell to 0.3686 — a bias of about 8%
  of the truth. Real, modest, and not knowable in advance without the
  diagnostics.
- **`pesaran_cd` and `alpha_exponent` can disagree, and the disagreement is
  informative.** On both residual sets CD rejected weak dependence while the
  exponent could not be identified at all. Only the raw strong-factor series
  gave `alpha = 0.980`, which is the `alpha = 1` a pervasive factor implies, and
  pure noise was refused like the residuals. A rejection from CD is not evidence
  of a factor.
- `alpha_exponent` throwing is not a failure to handle a case. Returning a
  number below 0.5 would report a value for a parameter that does not exist, and
  more data would not help. Treat the exception as the finding.
- `fit_sp_cce` recovered `rho` to +0.0101 where fixed effects gave -0.0314. The
  argument for it is cheap insurance against a bias of unknown size, not rescue
  from disaster.